In [ ]:
# -*- coding:utf-8 -*-
"""
walk_map_minfirst_loop.py
รวม Reactive single-scan (L/F/R/B) + Min-first/priority เข้ากับ DFS Explorer

นโยบายรอบลูป (แก้ตามที่ขอ):
  1) สแกน L/F/R/B หนึ่งครั้ง → ตัดสินใจ Reactive (ทำอย่างมาก 1 แอคชัน) "ครั้งเดียวพอ"
  2) หน่วงสั้น ๆ ให้หุ่นนิ่ง
  3) ทำ DFS 1 สเต็ป (ไปเซลล์ใหม่หรือ backtrack 1 ก้าว)
  4) วนซ้ำ
"""

import time
import csv
import math
from typing import Dict, Tuple, Optional
from robomaster import robot

# ===================== CONFIG (ทั่วไป) =====================
CONN_TYPE = "ap"
FREQ_HZ = 10                 # subscribe ToF
SCAN_SETTLE_S = 0.10         # หน่วงหลังหมุนกิมบอลเพื่อให้ค่า ToF นิ่งขึ้น
READ_TIMEOUT = 1.0           # รออ่าน ToF สูงสุดหลังหมุน
GIMBAL_SPEED = 180           # °/s
REACTIVE_SETTLE_S = 0.08     # หน่วงสั้น ๆ หลัง reactive ก่อนเข้า DFS
LOOP_SLEEP = 0.05            # เวลาพักเล็กน้อยตอนจบลูป

# ===================== CONFIG (Reactive L/F/R/B) =====================
# เกณฑ์ (เมตร)
TOO_CLOSE = 0.15                           # ใกล้กำแพงเกินไป → ต้องถอย/สไลด์ออก
TOO_FAR_LO, TOO_FAR_HI = 0.20, 0.30        # ห่างเกินไป (ช่วงที่อยาก “ขยับเข้า” ข้าง/หลัง)
CLEAR_ALL = 0.40                           # เคลียร์ทุกทิศ -> ไม่ต้องทำอะไร
QUIET_BAND_LO, QUIET_BAND_HI = 0.16, 0.19  # ทุกทิศอยู่ในแคบ ๆ นี้ -> ไม่ทำอะไร

STEP_M = 0.10              # ระยะก้าวแก้ชิด/ห่าง (Reactive)
XY_SPEED = 1.0

# มุมกิมบอล (องศา) สำหรับ L/F/R/B
ANGLES_LFRB = {"ซ้าย": -90, "หน้า": 0, "ขวา": 90, "หลัง": 180}

# ===================== CONFIG (DFS) =====================
CELL_SIZE = 0.6             # ระยะเซลล์กริด (เมตร)
X_SPEED_DEFAULT = 0.18      # ความเร็ว slide (m/s) แบบ open-loop
TOF_THRESHOLD_MM = 600      # ถ้า ToF < 600 mm ถือว่าเป็นกำแพง
TOF_MAX_VALID_MM = 3000     # ค่าสูงสุดที่ถือว่ายัง valid
TOF_STALE_TIMEOUT = 1.0     # ค่าที่อ่านเกินเวลานี้จะไม่นำมาใช้
CSV_FILENAME = "map_data.csv"

# ทิศบน "กริด" และ heading ของการขับ (0° = เดินหน้าตามแกน x ของแชสซี)
# หมายเหตุ: ชื่อ N/E/S/W คือ "กริด" ไม่ใช่แกนหุ่น
DIRECTIONS = {
    "N": (0.0,  0, +1, 0.0),    # (gimbal_yaw_for_scan, dx, dy, drive_heading_deg)
    "E": (90.0, +1, 0, 90.0),
    "S": (180.0, 0, -1, 180.0),
    "W": (-90.0, -1, 0, -90.0),
}

# ===================== STATE =====================
map_data: Dict[Tuple[int, int], Dict] = {}
pos = (0, 0)            # ตำแหน่งกริดปัจจุบัน
stack = []              # ใช้สำหรับ backtrack แบบ DFS

pose_x = 0.0            # โอโดเมทรี (เมตร)
pose_y = 0.0
pose_yaw = 0.0          # องศา

last_tof_mm: Optional[float] = None
last_tof_time: float = 0.0
last_gimbal_yaw: Optional[float] = None

# ===================== SUBSCRIBE CALLBACKS =====================
def sub_position_handler(info):
    """อัปเดตโอโดเมทรีจากแชสซี (x, y, yaw)"""
    global pose_x, pose_y, pose_yaw
    try:
        if isinstance(info, (list, tuple)):
            pose_x, pose_y = float(info[0]), float(info[1])
            if len(info) > 2:
                pose_yaw = float(info[2])
        elif isinstance(info, dict):
            pose_x = float(info.get("x", pose_x))
            pose_y = float(info.get("y", pose_y))
            pose_yaw = float(info.get("yaw", info.get("theta", pose_yaw)))
    except Exception:
        pass

def tof_handler(info):
    """อัปเดต ToF1 (ช่องแรก) หน่วย mm"""
    global last_tof_mm, last_tof_time
    try:
        val = info[0] if isinstance(info, (list, tuple)) else info.get("distance", info)
        if val is not None and 0 < val < TOF_MAX_VALID_MM:
            last_tof_mm = float(val)
            last_tof_time = time.time()
        else:
            last_tof_mm = None
    except Exception:
        last_tof_mm = None

# ===================== UTILS =====================
def mm_to_m(mm: Optional[float]) -> float:
    return float('nan') if mm is None else mm / 1000.0

def normalize_angle(angle_deg):
    return ((angle_deg + 180.0) % 360.0) - 180.0

def set_cell(x, y):
    """สร้างเซลล์ในแผนที่ถ้ายังไม่มี"""
    if (x, y) not in map_data:
        map_data[(x, y)] = {
            "walls": dict.fromkeys("NESW", False),
            "visited": False
        }

def save_map():
    """บันทึกแผนที่และสถานะสุดท้ายลง CSV"""
    try:
        with open(CSV_FILENAME, "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["x","y","wall_N","wall_E","wall_S","wall_W","visited","pose_x","pose_y","pose_yaw"])
            for (x, y), cell in sorted(map_data.items()):
                walls = cell["walls"]
                w.writerow([x, y, walls["N"], walls["E"], walls["S"], walls["W"],
                            cell["visited"], pose_x, pose_y, pose_yaw])
        print(f"[INFO] Map saved to {CSV_FILENAME}")
    except Exception as e:
        print(f"[ERROR] Failed to save map: {e}")

# ===================== MOTION (DFS ใช้) =====================
def move_slide(chassis, dist, direction_deg, speed=X_SPEED_DEFAULT):
    """
    Slide แบบ open-loop: เลื่อนไปตามมุม absolute 'direction_deg' ระยะ 'dist' ที่ความเร็ว 'speed'
    0° = เดินหน้าตามแกน x ของแชสซี, 90° = ไปทางขวาแกน y
    """
    if dist == 0 or speed == 0:
        return
    duration = abs(dist / speed)
    x_speed = speed * math.cos(math.radians(direction_deg))
    y_speed = speed * math.sin(math.radians(direction_deg))
    start = time.time()
    while time.time() - start < duration:
        chassis.drive_speed(x_speed, y_speed, 0.0)
        time.sleep(0.05)
    chassis.drive_speed(0.0, 0.0, 0.0)
    time.sleep(0.05)

# ===================== GIMBAL / SCAN =====================
def gimbal_to(ep_gimbal, yaw):
    """หมุนกิมบอลไป yaw ที่กำหนด พร้อมหน่วงให้ ToF นิ่ง"""
    global last_gimbal_yaw
    try:
        if last_gimbal_yaw is None or abs(last_gimbal_yaw - yaw) > 1.0:
            ep_gimbal.moveto(pitch=0, yaw=yaw, yaw_speed=GIMBAL_SPEED).wait_for_completed()
            last_gimbal_yaw = yaw
        time.sleep(SCAN_SETTLE_S)
    except Exception:
        try:
            ep_gimbal.moveto(pitch=0, yaw=yaw)
        except Exception:
            pass
        last_gimbal_yaw = yaw
        time.sleep(SCAN_SETTLE_S + 0.02)

def scan_lfrb_once(ep_gimbal) -> Dict[str, float]:
    """
    สแกน 4 ทิศ L/F/R/B ครั้งเดียว โดยอ่าน ToF1 ที่ subscribe ไว้
    คืนค่าเป็นเมตร { 'ซ้าย':m, 'หน้า':m, 'ขวา':m, 'หลัง':m } (NaN ถ้าอ่านไม่ได้ทันเวลา)
    """
    out: Dict[str, float] = {}
    for name, yaw in ANGLES_LFRB.items():
        gimbal_to(ep_gimbal, yaw)
        # รอค่าใหม่เล็กน้อย (หรือใช้ค่าล่าสุดถ้ายังสด)
        start = time.time()
        got = None
        while time.time() - start < READ_TIMEOUT:
            if last_tof_mm is not None and (time.time() - last_tof_time) < TOF_STALE_TIMEOUT:
                got = last_tof_mm
                break
            time.sleep(0.01)
        out[name] = mm_to_m(got)
    return out

def print_scan_lfrb(res: Dict[str, float]):
    def fmt(v): return ("---" if (v != v) else f"{v:.3f}")  # NaN -> '---'
    print(f"ซ้าย {fmt(res['ซ้าย'])} หน้า {fmt(res['หน้า'])} ขวา {fmt(res['ขวา'])} หลัง {fmt(res['หลัง'])}")

# ===================== Reactive DECISION (Min-first + Priority) =====================
def try_act_for_dir_min(ep_chassis, d: Dict[str, float]) -> bool:
    """ทำแอคชันเฉพาะ 'ด้านที่ระยะต่ำสุด' ถ้าเข้าเกณฑ์"""
    safe_d = {k: (d[k] if d[k] == d[k] else 999.0) for k in d}  # NaN → ใหญ่ ๆ
    dir_min = min(safe_d, key=lambda k: safe_d[k])
    v = safe_d[dir_min]
    print(f"[Min-first] ต่ำสุด: {dir_min} = {v:.3f} m")

    if dir_min == "ซ้าย":
        if v < TOO_CLOSE:
            print("  -> สไลด์ขวา 0.10 m")
            ep_chassis.move(x=0, y=+STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True
        if TOO_FAR_LO <= v <= TOO_FAR_HI:
            print("  -> สไลด์ซ้าย 0.10 m")
            ep_chassis.move(x=0, y=-STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    elif dir_min == "ขวา":
        if v < TOO_CLOSE:
            print("  -> สไลด์ซ้าย 0.10 m")
            ep_chassis.move(x=0, y=-STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True
        if TOO_FAR_LO <= v <= TOO_FAR_HI:
            print("  -> สไลด์ขวา 0.10 m")
            ep_chassis.move(x=0, y=+STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    elif dir_min == "หน้า":
        if v < TOO_CLOSE:
            print("  -> ถอยหลัง 0.10 m")
            ep_chassis.move(x=-STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    elif dir_min == "หลัง":
        if v < TOO_CLOSE:
            print("  -> เดินหน้า 0.10 m")
            ep_chassis.move(x=+STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True
        if TOO_FAR_LO <= v <= TOO_FAR_HI:
            print("  -> ถอยหลัง 0.10 m")
            ep_chassis.move(x=-STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    return False

def decide_and_act_with_priority(ep_chassis, d: Dict[str, float]) -> bool:
    """
    ทำอย่างมาก 1 แอคชันตามกฎ:
      A) Global no-op (ทุกทิศ 0.16–0.19 หรือทุกทิศ >= 0.4) -> none
      B) Min-first
      C) Priority chain
    คืน True ถ้าทำแอคชันแล้ว, False ถ้าไม่ทำอะไร
    """
    vals = list(v for v in d.values() if v == v)  # เอาเฉพาะที่ไม่ใช่ NaN
    if len(vals) == 4:
        if all(QUIET_BAND_LO <= v <= QUIET_BAND_HI for v in vals):
            print("[Reactive] none (ทุกทิศ 0.16–0.19 m)"); return False
        if all(v >= CLEAR_ALL for v in vals):
            print("[Reactive] none (เคลียร์ทุกทิศ >= 0.4 m)"); return False

    if try_act_for_dir_min(ep_chassis, d):
        return True

    def is_between(v, lo, hi): return (v == v) and (lo <= v <= hi)
    def is_lt(v, th):         return (v == v) and (v < th)

    if is_lt(d.get("ซ้าย", float('nan')), TOO_CLOSE):
        print("[Priority] ซ้ายใกล้ -> สไลด์ขวา 0.10 m")
        ep_chassis.move(x=0, y=+STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_lt(d.get("ขวา", float('nan')), TOO_CLOSE):
        print("[Priority] ขวาใกล้ -> สไลด์ซ้าย 0.10 m")
        ep_chassis.move(x=0, y=-STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_between(d.get("ซ้าย", float('nan')), TOO_FAR_LO, TOO_FAR_HI):
        print("[Priority] ซ้ายห่าง -> สไลด์ซ้าย 0.10 m")
        ep_chassis.move(x=0, y=-STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_between(d.get("ขวา", float('nan')), TOO_FAR_LO, TOO_FAR_HI):
        print("[Priority] ขวาห่าง -> สไลด์ขวา 0.10 m")
        ep_chassis.move(x=0, y=+STEP_M, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_lt(d.get("หน้า", float('nan')), TOO_CLOSE):
        print("[Priority] หน้าใกล้ -> ถอยหลัง 0.10 m")
        ep_chassis.move(x=-STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_lt(d.get("หลัง", float('nan')), TOO_CLOSE):
        print("[Priority] หลังใกล้ -> เดินหน้า 0.10 m")
        ep_chassis.move(x=+STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    if is_between(d.get("หลัง", float('nan')), TOO_FAR_LO, TOO_FAR_HI):
        print("[Priority] หลังห่าง -> ถอยหลัง 0.10 m")
        ep_chassis.move(x=-STEP_M, y=0, z=0, xy_speed=XY_SPEED).wait_for_completed(); return True

    print("[Reactive] none (ไม่เข้าเงื่อนไข)")
    return False

# ===================== DFS SCAN / UPDATE WALLS =====================
def scan_cell_walls(ep_gimbal, x, y):
    set_cell(x, y)
    walls = map_data[(x, y)]["walls"]
    for dir_name, (gimbal_yaw, dx, dy, _) in DIRECTIONS.items():
        gimbal_to(ep_gimbal, gimbal_yaw)
        dist = last_tof_mm if (time.time() - last_tof_time) < TOF_STALE_TIMEOUT else None
        if dist is None:
            walls[dir_name] = True
            print(f"[SCAN] ({x},{y}) {dir_name}: --- -> WALL")
        else:
            walls[dir_name] = (dist < TOF_THRESHOLD_MM)
            print(f"[SCAN] ({x},{y}) {dir_name}: {dist:.0f} mm -> {'WALL' if walls[dir_name] else 'OPEN'}")

# ===================== DFS STEP (ทำทีละสเต็ป) =====================
def dfs_one_step(ep_chassis, ep_gimbal) -> bool:
    global pos, stack
    x, y = pos
    set_cell(x, y)
    map_data[(x, y)]["visited"] = True

    scan_cell_walls(ep_gimbal, x, y)

    moved = False
    for dir_name, (_, dx, dy, heading_deg) in DIRECTIONS.items():
        nx, ny = x + dx, y + dy
        set_cell(nx, ny)
        if not map_data[(x, y)]["walls"][dir_name] and not map_data[(nx, ny)]["visited"]:
            print(f"[DFS] {pos} -> {(nx, ny)} via {dir_name}")
            move_slide(ep_chassis, CELL_SIZE, direction_deg=heading_deg, speed=X_SPEED_DEFAULT)
            stack.append((x, y, dir_name))
            pos = (nx, ny)
            moved = True
            break

    if moved:
        return True

    if not stack:
        print("[DFS] Complete (no more moves).")
        return False

    px, py, came_dir = stack.pop()
    opp_dir = {"N":"S","S":"N","E":"W","W":"E"}[came_dir]
    opp_heading = DIRECTIONS[opp_dir][3]
    print(f"[DFS-Back] {pos} -> {(px, py)} via {opp_dir}")
    move_slide(ep_chassis, CELL_SIZE, direction_deg=opp_heading, speed=X_SPEED_DEFAULT)
    pos = (px, py)
    return True

# ===================== MAIN LOOP =====================
def main():
    bot = robot.Robot()
    bot.initialize(conn_type=CONN_TYPE)

    chassis = bot.chassis
    gimbal  = bot.gimbal
    sensor  = bot.sensor

    chassis.sub_position(freq=10, callback=sub_position_handler)
    sensor.sub_distance(freq=FREQ_HZ, callback=tof_handler)

    gimbal.moveto(pitch=0, yaw=0, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(0.2)

    print("[START] Reactive(once) + DFS loop (Ctrl+C เพื่อหยุด)")
    try:
        while True:
            # 1) เช็คชิดกำแพง/แคบ ครั้งเดียว
            dists_m = scan_lfrb_once(gimbal)
            print_scan_lfrb(dists_m)
            did_reactive = decide_and_act_with_priority(chassis, dists_m)

            # หน่วงสั้น ๆ ให้นิ่ง แล้ว "ไปสเต็ปถัดไป" ทันที (ไม่วน reactive ซ้ำ)
            time.sleep(REACTIVE_SETTLE_S)

            # 2) DFS 1 สเต็ป
            did_dfs = dfs_one_step(chassis, gimbal)
            if not did_dfs:
                print("[DONE] Exploration complete. Stop loop.")
                break

            time.sleep(LOOP_SLEEP)

    except KeyboardInterrupt:
        print("[STOP] Interrupted by user")
    finally:
        try: sensor.unsub_distance()
        except Exception: pass
        try: chassis.unsub_position()
        except Exception: pass
        try: gimbal.moveto(pitch=0, yaw=0, yaw_speed=GIMBAL_SPEED).wait_for_completed()
        except Exception: pass
        try: bot.close()
        except Exception: pass
        save_map()

if _name_ == "_main_":
    main()

## ใหม่

In [ ]:
# -*- coding:utf-8 -*-
# สแกนมุมเอียง (-45, 45, -135, 135) แล้วทำตามลำดับความสำคัญ (ทำครั้งเดียวแล้วจบ)
# เพิ่ม: ฟังก์ชันตัดสินใจแบบใหม่ (min-first) + วิ่งลูป: ใหม่ก่อน ถ้าไม่ได้ค่อยใช้แบบเก่า

import time
from math import isfinite
from robomaster import robot

# ===== Tunables =====
CONN_TYPE = "ap"      # หรือ "sta" ตามการเชื่อมต่อของคุณ
FREQ_HZ = 10
DWELL_SEC = 0.20      # หน่วงให้ลำแสงนิ่งหลังหมุน
READ_TIMEOUT = 1.0
GIMBAL_SPEED = 180
THRESH = 0.20         # เกณฑ์ (m)
SLIDE_DIST = 0.10     # ระยะสไลด์โดยประมาณ (m)
WHEEL_SPEED = 30      # ความเร็วล้อ (ปรับตามจริง)

# มุมเอียง (มาตรฐาน: ซ้าย = ลบ, ขวา = บวก)
ANGLES = {
    "เอียงหน้าซ้าย":  -45,
    "เอียงหน้าขวา":    45,
    "เอียงหลังซ้าย": -135,
    "เอียงหลังขวา":   135,
}

# ลำดับความสำคัญ (ตรรกะเก่า)
PRIORITY = ["เอียงหน้าซ้าย", "เอียงหน้าขวา", "เอียงหลังซ้าย", "เอียงหลังขวา"]

_last_tof1_mm = None  # อัปเดตจาก callback

def _tof_cb(sub_info):
    """sub_info = [tof1, tof2, tof3, tof4] หน่วย mm"""
    global _last_tof1_mm
    _last_tof1_mm = sub_info[0]

def mm_to_m(mm):
    if mm is None:
        return float('nan')
    m = mm / 1000.0
    return m if m <= 10.0 else float('nan')

def gimbal_moveto_and_read(ep_gimbal, yaw_deg):
    """หมุนกิมบอลถึงมุมก่อน แล้วค่อยอ่านค่า ToF1 (m)"""
    global _last_tof1_mm
    _last_tof1_mm = None  # ล้างค่าของมุมก่อนหน้า
    ep_gimbal.moveto(pitch=0, yaw=yaw_deg, yaw_speed=GIMBAL_SPEED).wait_for_completed()
    time.sleep(DWELL_SEC)
    start = time.time()
    while time.time() - start < READ_TIMEOUT:
        if _last_tof1_mm is not None:
            return mm_to_m(_last_tof1_mm)
        time.sleep(0.01)
    return float('nan')

# ===== การสไลด์ล้อเฉพาะคู่ =====
def _sleep_for_distance(speed_abs, dist):
    # สมมติฐานคร่าว ๆ: เวลา ~ dist / (speed_abs/100). ควรคาลิเบรตจริง
    return dist / max(1e-6, (speed_abs / 100.0))

def slide_front(ep_chassis, to_right: bool):
    """สไลด์เฉพาะ 2 ล้อหน้า: True=ไปขวา, False=ไปซ้าย"""
    sp = WHEEL_SPEED if to_right else -WHEEL_SPEED
    ep_chassis.drive_wheels(sp, -sp, 0, 0)
    time.sleep(_sleep_for_distance(abs(sp), SLIDE_DIST))
    ep_chassis.drive_wheels(0, 0, 0, 0)

def slide_back(ep_chassis, to_right: bool):
    """สไลด์เฉพาะ 2 ล้อหลัง: True=ไปขวา, False=ไปซ้าย"""
    sp = WHEEL_SPEED if to_right else -WHEEL_SPEED
    ep_chassis.drive_wheels(0, 0, sp, -sp)
    time.sleep(_sleep_for_distance(abs(sp), SLIDE_DIST))
    ep_chassis.drive_wheels(0, 0, 0, 0)

# ---------------------- ฟังก์ชันใหม่: สแกนครั้งเดียว ----------------------
def scan_all_angles(ep_gimbal):
    """สแกนทุกมุมเอียงครั้งเดียว คืน dict ชื่อมุม -> ระยะเมตร (NaN ถ้าอ่านไม่ทัน)"""
    dists_m = {}
    for name, yaw in ANGLES.items():
        d_m = gimbal_moveto_and_read(ep_gimbal, yaw)
        dists_m[name] = d_m
    return dists_m

def print_scan(dists_m):
    out = []
    for name in ANGLES.keys():
        v = dists_m.get(name, float('nan'))
        out.append(f"{name}: {'---' if not isfinite(v) else f'{v:.3f} m'}")
    print(" | ".join(out))

# ---------------------- ฟังก์ชันใหม่: ตัดสินใจแบบ “ใหม่ก่อน/เก่าตาม” ----------------------
def _act_for_label(ep_chassis, label):
    """แมปชื่อมุม -> แอคชันเดิมที่คุณกำหนด"""
    if label == "เอียงหน้าซ้าย":
        slide_front(ep_chassis, to_right=True)
        return True
    if label == "เอียงหน้าขวา":
        slide_front(ep_chassis, to_right=False)
        return True
    if label == "เอียงหลังซ้าย":
        slide_back(ep_chassis, to_right=True)
        return True
    if label == "เอียงหลังขวา":
        slide_back(ep_chassis, to_right=False)
        return True
    return False

def decide_new_minfirst(ep_chassis, dists_m):
    """
    ตัดสินใจแบบใหม่: เลือก "มุมที่ระยะน้อยสุด" แล้วถ้า <= THRESH ให้ทำแอคชันตาม label นั้น
    """
    # เลือกเฉพาะค่าที่เป็นตัวเลขได้
    finite_items = [(k, v) for k, v in dists_m.items() if isfinite(v)]
    if not finite_items:
        return False
    # หา min
    label_min, val_min = min(finite_items, key=lambda kv: kv[1])
    print(f"[NEW] ต่ำสุด: {label_min} = {val_min:.3f} m")
    if val_min <= THRESH:
        print(f"[NEW] ทำแอคชันตามมุมต่ำสุด: {label_min}")
        return _act_for_label(ep_chassis, label_min)
    return False

def decide_old_priority(ep_chassis, dists_m):
    """
    ตัดสินใจแบบเก่า: เดินตาม PRIORITY เดิม เจออันแรกที่ <= THRESH ก็ทำแล้วจบ
    """
    for name in PRIORITY:
        if name not in dists_m:
            print(f"[OLD] ข้าม {name} (ไม่มีข้อมูล)")
            continue
        val = dists_m[name]
        if not isfinite(val):
            continue
        if val <= THRESH:
            print(f"[OLD] เงื่อนไขตรง: {name} ({val:.3f} m)")
            return _act_for_label(ep_chassis, name)
    return False

def main():
    ep = robot.Robot()
    ep.initialize(conn_type=CONN_TYPE)

    ep_gimbal = ep.gimbal
    ep_chassis = ep.chassis
    ep_sensor = ep.sensor

    # ตั้งกิมบอลศูนย์ก่อนเริ่ม
    ep_gimbal.moveto(pitch=0, yaw=0, yaw_speed=GIMBAL_SPEED).wait_for_completed()

    # เปิดรับ ToF
    ep_sensor.sub_distance(freq=FREQ_HZ, callback=_tof_cb)
    time.sleep(0.2)

    try:
        print("[START] loop: ตัดสินใจใหม่ก่อน -> ถ้าไม่ได้ ค่อยใช้แบบเก่า (Ctrl+C เพื่อหยุด)")
        while True:
            # 1) สแกนมุมเอียงทั้งหมด (ครั้งเดียว)
            dists_m = scan_all_angles(ep_gimbal)
            print_scan(dists_m)

            # 2) ตัดสินใจแบบใหม่ (min-first) ก่อน
            acted = decide_new_minfirst(ep_chassis, dists_m)

            # 3) ถ้ายังไม่ได้ทำ -> ตกมาใช้ตรรกะเก่า (PRIORITY เดิม)
            if not acted:
                acted = decide_old_priority(ep_chassis, dists_m)

            if not acted:
                print("[INFO] ไม่มีมุมใดต่ำกว่าเกณฑ์ -> ไม่ทำอะไร")

            # 4) หน่วงสั้น ๆ แล้ววนลูป
            time.sleep(0.08)

    except KeyboardInterrupt:
        print("[STOP] Interrupted by user")
    finally:
        try:
            ep_sensor.unsub_distance()
        except Exception:
            pass
        try:
            ep_gimbal.moveto(pitch=0, yaw=0, yaw_speed=GIMBAL_SPEED).wait_for_completed()
        except Exception:
            pass
        ep.close()
        print("Done.")

if __name__ == "__main__":
    main()


[START] Looping (Ctrl+C เพื่อหยุด)
=== สแกน L/F/R/B (m) ===
ซ้าย 0.148 หน้า 0.839 ขวา 0.146 หลัง 0.197
[New-Min] ต่ำสุด: ขวา = 0.146 m
 -> สไลด์ซ้าย 0.10 m
=== สแกนมุมเอียง (m) ===
เอียง: เอียงหน้าซ้าย 0.237 m, เอียงหน้าขวา 0.385 m, เอียงหลังซ้าย 0.173 m, เอียงหลังขวา 0.379 m
[Old] เงื่อนไขตรง: เอียงหลังซ้าย (0.173 m)
 -> ล้อหลัง สไลด์ขวา 0.10 m
=== สแกน L/F/R/B (m) ===
ซ้าย 0.064 หน้า 0.843 ขวา 0.285 หลัง 0.198
[New-Min] ต่ำสุด: ซ้าย = 0.064 m
 -> สไลด์ขวา 0.10 m
=== สแกนมุมเอียง (m) ===
เอียง: เอียงหน้าซ้าย 0.932 m, เอียงหน้าขวา 0.223 m, เอียงหลังซ้าย 0.167 m, เอียงหลังขวา 0.247 m
[Old] เงื่อนไขตรง: เอียงหลังซ้าย (0.167 m)
 -> ล้อหลัง สไลด์ขวา 0.10 m
=== สแกน L/F/R/B (m) ===
ซ้าย 0.123 หน้า 0.888 ขวา 0.226 หลัง 0.181
[New-Min] ต่ำสุด: ซ้าย = 0.123 m
 -> สไลด์ขวา 0.10 m
=== สแกนมุมเอียง (m) ===
เอียง: เอียงหน้าซ้าย 0.989 m, เอียงหน้าขวา 0.152 m, เอียงหลังซ้าย 0.169 m, เอียงหลังขวา 0.118 m
[Old] เงื่อนไขตรง: เอียงหน้าขวา (0.152 m)
 -> ล้อหน้า สไลด์ซ้าย 0.10 m
[STOP] Interrupted by user